# Copycat

Copycat is a small concatenative language in which a model invocation is a
first-class program-synthesis effect. The deterministic runtime stays in
control: `{natural language}` asks a backend for Copycat code, and that code
continues immediately against the current data stack.

The notebook is deliberately organized for surgical editing. Every top-level
class and function has its own cell (tests are the sole exception).


## Setup — run this section first

This section contains all dependencies and definitions needed to use Copycat.
In Google Colab, **Run section** on this heading initializes the complete
language and the optional Gemma backend without loading model weights.


### Dependencies


In [ ]:
%pip install -q -U "transformers>=5.5.0" accelerate lark ipytest


In [ ]:
from __future__ import annotations

import dataclasses
import json
import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from typing import Callable, Mapping, Optional, Protocol, Sequence

from lark import Lark, Transformer
from lark.exceptions import UnexpectedCharacters, UnexpectedInput, UnexpectedToken


### Runtime objects


In [ ]:
@dataclass(frozen=True)
class Span:
    source: str
    source_name: str
    start: int
    end: int
    line: int
    column: int
    end_line: int
    end_column: int

    @classmethod
    def from_token(cls, token, source: str, source_name: str) -> "Span":
        return cls(
            source=source,
            source_name=source_name,
            start=token.start_pos,
            end=token.end_pos,
            line=token.line,
            column=token.column,
            end_line=token.end_line,
            end_column=token.end_column,
        )

    def context(self) -> str:
        lines = self.source.splitlines() or [""]
        line_index = min(max(self.line - 1, 0), len(lines) - 1)
        text = lines[line_index]
        width = max(1, (self.end - self.start) if self.line == self.end_line else 1)
        width = min(width, max(1, len(text) - self.column + 2))
        return f"{text}\n{' ' * (self.column - 1)}{'^' * width}"


In [ ]:
@dataclass(frozen=True, kw_only=True)
class Object:
    span: Optional[Span] = field(default=None, compare=False, repr=False)


In [ ]:
@dataclass(frozen=True)
class Word(Object):
    name: str

    def __str__(self) -> str:
        return self.name


In [ ]:
@dataclass(frozen=True)
class Abs(Object):
    body: Object

    def __str__(self) -> str:
        return f"[{self.body}]"


In [ ]:
@dataclass(frozen=True)
class Cat(Object):
    body: tuple[Object, ...]

    def __str__(self) -> str:
        return " ".join(str(child) for child in self.body)


In [ ]:
@dataclass(frozen=True)
class String(Object):
    value: str

    def __str__(self) -> str:
        return json.dumps(self.value, ensure_ascii=False)


In [ ]:
@dataclass(frozen=True)
class Number(Object):
    value: int

    def __str__(self) -> str:
        return str(self.value)


In [ ]:
@dataclass(frozen=True)
class Model(Object):
    prompt: str

    def __str__(self) -> str:
        return "{" + self.prompt + "}"


### Structured conditions


In [ ]:
class CopycatError(Exception):
    """Base class for errors intended for humans and future synthesis loops."""


In [ ]:
@dataclass
class ParseError(CopycatError):
    message: str
    source: str
    source_name: str
    line: int
    column: int
    expected: tuple[str, ...] = ()

    def __str__(self) -> str:
        lines = self.source.splitlines() or [""]
        line_index = min(max(self.line - 1, 0), len(lines) - 1)
        text = lines[line_index]
        caret = " " * max(0, self.column - 1) + "^"

        parts = [
            f"Parse error in {self.source_name} at line {self.line}, column {self.column}:",
            self.message,
            "",
            text,
            caret,
        ]
        if self.expected:
            parts.extend(["", "Expected: " + ", ".join(self.expected)])
        return "\n".join(parts)


In [ ]:
@dataclass
class EvaluationError(CopycatError):
    message: str
    operation: Optional[Object] = None

    def __str__(self) -> str:
        if self.operation is None or self.operation.span is None:
            return f"Evaluation error: {self.message}"

        span = self.operation.span
        return (
            f"Evaluation error in {span.source_name} at line {span.line}, "
            f"column {span.column}:\n{self.message}\n\n{span.context()}"
        )


In [ ]:
@dataclass
class ModelProtocolError(CopycatError):
    answer: str
    message: str

    def __str__(self) -> str:
        return (
            "Model protocol error: "
            + self.message
            + "\n\nModel answer:\n"
            + self.answer
        )


In [ ]:
@dataclass
class ModelReportedError(CopycatError):
    prompt: str
    message: str

    def __str__(self) -> str:
        return f"Model reported an error for {{{self.prompt}}}: {self.message}"


In [ ]:
@dataclass
class GeneratedCodeError(CopycatError):
    prompt: str
    code: str
    cause: Exception

    def __str__(self) -> str:
        return (
            f"Generated code for {{{self.prompt}}} could not be used.\n\n"
            f"Generated code:\n{self.code}\n\n"
            f"Cause:\n{self.cause}"
        )


### Reader and parser


In [ ]:
_GRAMMAR = r"""
start: term*

?term: quotation
     | MODEL           -> model
     | ESCAPED_STRING  -> string
     | NUMBER          -> number
     | WORD            -> word

quotation: "[" term* "]"

MODEL.3: /\{[^}]*\}/s
NUMBER: /0|[1-9][0-9]*/
WORD: /[A-Za-z][A-Za-z0-9_]*/

%import common.ESCAPED_STRING
%import common.WS
%ignore WS
"""

_PARSER = Lark(_GRAMMAR, parser="lalr", propagate_positions=True)


In [ ]:
class _BuildAST(Transformer):
    def __init__(self, source: str, source_name: str):
        super().__init__()
        self.source = source
        self.source_name = source_name

    def _span(self, token) -> Span:
        return Span.from_token(token, self.source, self.source_name)

    def word(self, children):
        (token,) = children
        return Word(str(token), span=self._span(token))

    def number(self, children):
        (token,) = children
        return Number(int(str(token)), span=self._span(token))

    def string(self, children):
        (token,) = children
        return String(json.loads(str(token)), span=self._span(token))

    def model(self, children):
        (token,) = children
        text = str(token)
        return Model(text[1:-1], span=self._span(token))

    def quotation(self, children):
        return Abs(Cat(tuple(children)))

    def start(self, children):
        return Cat(tuple(children))


In [ ]:
def _eof_location(source: str) -> tuple[int, int]:
    lines = source.splitlines()
    if not lines:
        return 1, 1
    return len(lines), len(lines[-1]) + 1


In [ ]:
def _friendly_parse_error(
    exc: UnexpectedInput,
    source: str,
    source_name: str,
) -> ParseError:
    raw_expected = tuple(
        sorted(
            getattr(exc, "expected", ())
            or getattr(exc, "allowed", ())
            or ()
        )
    )

    terminal_names = {
        "ESCAPED_STRING": "string",
        "LSQB": "'['",
        "RSQB": "']'",
        "MODEL": "model invocation {...}",
        "NUMBER": "natural number",
        "WORD": "word",
        "$END": "end of input",
    }
    expected = tuple(terminal_names.get(name, name) for name in raw_expected)

    if isinstance(exc, UnexpectedCharacters):
        pos = exc.pos_in_stream
        char = source[pos : pos + 1]

        if char == "{":
            message = "Unclosed model invocation. Add a matching '}'."
        elif char == '"':
            message = "Invalid or unterminated string literal."
        else:
            shown = repr(char) if char else "end of input"
            message = f"Unexpected character {shown}."

        return ParseError(
            message, source, source_name, exc.line, exc.column, expected
        )

    if isinstance(exc, UnexpectedToken):
        token = exc.token

        if token.type == "$END":
            line, column = _eof_location(source)
            if "RSQB" in raw_expected:
                message = "Unclosed quotation. Add a matching ']'."
            else:
                message = "Unexpected end of input."

            return ParseError(
                message, source, source_name, line, column, expected
            )

        if token.type == "RSQB":
            message = "Unexpected ']'. There is no matching '[' for this bracket."
        else:
            message = f"Unexpected token {str(token)!r}."

        return ParseError(
            message, source, source_name, exc.line, exc.column, expected
        )

    return ParseError(
        "Could not parse input.",
        source,
        source_name,
        exc.line,
        exc.column,
        expected,
    )


In [ ]:
def read(code: str, *, source_name: str = "<input>") -> Cat:
    """Parse Copycat source into an AST."""
    try:
        tree = _PARSER.parse(code)
    except UnexpectedInput as exc:
        raise _friendly_parse_error(exc, code, source_name) from None

    return _BuildAST(code, source_name).transform(tree)


### Model response protocol


In [ ]:
@dataclass(frozen=True)
class ModelTurn:
    answer: str
    thinking: Optional[str] = None
    raw: Optional[str] = None
    streamed: bool = False


In [ ]:
class ModelBackend(Protocol):
    def generate(self, *, prompt: str, stack: str) -> ModelTurn:
        ...


In [ ]:
@dataclass(frozen=True)
class ModelOK:
    code: str


In [ ]:
@dataclass(frozen=True)
class ModelError:
    message: str


In [ ]:
_PROTOCOL_ELEMENT = re.compile(
    r"<(?P<tag>OK|ERROR)>(?P<payload>.*?)</(?P=tag)>",
    re.IGNORECASE | re.DOTALL,
)

_PROTOCOL_TAG = re.compile(
    r"</?(?:OK|ERROR)\b[^>]*>",
    re.IGNORECASE,
)


In [ ]:
def parse_model_answer(answer: str) -> ModelOK | ModelError:
    """Extract one case-insensitive OK or ERROR element from a model answer."""
    matches = list(_PROTOCOL_ELEMENT.finditer(answer))
    tag_markers = list(_PROTOCOL_TAG.finditer(answer))

    if len(matches) != 1 or len(tag_markers) != 2:
        raise ModelProtocolError(
            answer,
            "Expected exactly one <OK>...</OK> or <ERROR>...</ERROR> "
            "element (tag names are case-insensitive). Text outside that "
            "single element is allowed.",
        )

    match = matches[0]
    tag = match.group("tag").upper()
    normalized = f"<{tag}>{match.group('payload')}</{tag}>"

    try:
        root = ET.fromstring(normalized)
    except ET.ParseError as exc:
        raise ModelProtocolError(
            answer,
            "The protocol element is not well-formed XML.",
        ) from exc

    if root.attrib or list(root):
        raise ModelProtocolError(
            answer,
            "Protocol elements may not have attributes or child elements.",
        )

    payload = root.text or ""

    if tag == "OK":
        return ModelOK(payload.strip())

    message = payload.strip()
    if not message:
        raise ModelProtocolError(
            answer,
            "<ERROR> must contain a useful error message.",
        )
    return ModelError(message)


In [ ]:
class StubModel:
    """Deterministic backend for parser/evaluator tests and examples."""

    def __init__(self, response: str | Callable[[str, str], str]):
        self.response = response
        self.calls: list[tuple[str, str]] = []

    def generate(self, *, prompt: str, stack: str) -> ModelTurn:
        self.calls.append((prompt, stack))

        if callable(self.response):
            answer = self.response(prompt, stack)
        else:
            answer = self.response

        return ModelTurn(answer=answer, thinking="stub")


### Evaluator state


In [ ]:
Primitive = Callable[["State"], None]


In [ ]:
@dataclass
class State:
    gas: int
    code: list[Object]
    data: list[Object]
    sink: list[Object]
    dictionary: Mapping[str, Object]
    primitives: Mapping[str, Primitive]
    model_backend: Optional[ModelBackend]
    strict: bool
    allow_nested_model_calls: bool
    verbose: bool
    hand: Optional[Object] = None
    step: int = 0

    @property
    def value(self) -> Object:
        return Cat(tuple(self.sink + self.data + list(reversed(self.code))))

    @property
    def is_active(self) -> bool:
        return self.gas > 0 and bool(self.code)

    def tick(self) -> Object:
        self.gas -= 1
        self.step += 1
        self.hand = self.code.pop()
        return self.hand

    def trace(self, phase: str, term: Object) -> None:
        if not self.verbose:
            return

        sink = str(Cat(tuple(self.sink))) or "(empty)"
        data = str(Cat(tuple(self.data))) or "(empty)"
        remaining = str(Cat(tuple(reversed(self.code)))) or "(empty)"
        print(
            f"\n[Copycat step {self.step} — {phase}] "
            f"{type(term).__name__}: {term}"
        )
        print(f"  gas remaining: {self.gas}")
        print(f"  sink: {sink}")
        print(f"  data (bottom to top): {data}")
        print(f"  remaining code: {remaining}")

    def thunk(self) -> None:
        self.sink.extend(self.data)
        self.data = []
        if self.hand is not None:
            self.sink.append(self.hand)

    def stop(self) -> None:
        self.thunk()
        self.gas = 0

    def fail_or_thunk(self, message: str, *, stop: bool = False) -> None:
        if self.verbose:
            action = "error" if self.strict else ("stop" if stop else "residualize")
            print(f"\n[Copycat {action}] {message}")

        if self.strict:
            raise EvaluationError(message, self.hand)

        if stop:
            self.stop()
        else:
            self.thunk()


In [ ]:
def stack_string(data: Sequence[Object]) -> str:
    """Representation shown to the model, ordered bottom-to-top."""
    return str(Cat(tuple(data)))


In [ ]:
def _contains_model(obj: Object) -> bool:
    match obj:
        case Model():
            return True
        case Abs(body):
            return _contains_model(body)
        case Cat(body):
            return any(_contains_model(child) for child in body)
        case _:
            return False


### Primitive combinators


In [ ]:
def op_copy(state: State) -> None:
    if not state.data:
        state.fail_or_thunk(
            "Copy needs 1 value on the data stack, but the stack is empty."
        )
        return
    state.data.append(state.data[-1])


In [ ]:
def op_drop(state: State) -> None:
    if not state.data:
        state.fail_or_thunk(
            "Drop needs 1 value on the data stack, but the stack is empty."
        )
        return
    state.data.pop()


In [ ]:
def op_swap(state: State) -> None:
    if len(state.data) < 2:
        state.fail_or_thunk(
            f"Swap needs 2 values on the data stack, but found {len(state.data)}."
        )
        return
    state.data[-2], state.data[-1] = state.data[-1], state.data[-2]


In [ ]:
def op_abs(state: State) -> None:
    if not state.data:
        state.fail_or_thunk(
            "Abs needs 1 value on the data stack, but the stack is empty."
        )
        return
    state.data[-1] = Abs(state.data[-1])


In [ ]:
def op_app(state: State) -> None:
    if not state.data:
        state.fail_or_thunk(
            "App needs a quotation on top of the data stack.",
            stop=True,
        )
        return

    block = state.data[-1]

    if not isinstance(block, Abs):
        state.fail_or_thunk(
            f"App expected a quotation on top of the stack, but found {block}.",
            stop=True,
        )
        return

    state.data.pop()
    state.code.append(block.body)


In [ ]:
def _cat_objects(first: Object, second: Object) -> Cat:
    first_items = first.body if isinstance(first, Cat) else (first,)
    second_items = second.body if isinstance(second, Cat) else (second,)
    return Cat(tuple(first_items) + tuple(second_items))


In [ ]:
def op_cat(state: State) -> None:
    if len(state.data) < 2:
        state.fail_or_thunk(
            f"Cat needs 2 quotations on the data stack, but found {len(state.data)}."
        )
        return

    first, second = state.data[-2], state.data[-1]

    if not isinstance(first, Abs) or not isinstance(second, Abs):
        state.fail_or_thunk(
            f"Cat expected two quotations, but found {first} and {second}."
        )
        return

    state.data[-2:] = [Abs(_cat_objects(first.body, second.body))]


In [ ]:
def op_jump(state: State) -> None:
    if not state.data:
        state.fail_or_thunk(
            "Jump needs a handler quotation on top of the data stack.",
            stop=True,
        )
        return

    handler = state.data[-1]

    if not isinstance(handler, Abs):
        state.fail_or_thunk(
            f"Jump expected a handler quotation, but found {handler}.",
            stop=True,
        )
        return

    buffer: list[Object] = []
    index = 1
    mark_found = False

    while index <= len(state.code):
        point = state.code[-index]

        if isinstance(point, Word) and point.name == "Mark":
            mark_found = True
            break

        buffer.append(point)
        index += 1

    if not mark_found:
        state.fail_or_thunk(
            "Jump could not find a matching Mark in the continuation.",
            stop=True,
        )
        return

    continuation = Abs(Cat(tuple(buffer)))
    state.code = state.code[:-index]
    state.data.pop()
    state.data.append(continuation)
    state.code.append(handler.body)


In [ ]:
def op_mark(state: State) -> None:
    state.thunk()


### Evaluator


In [ ]:
def _run_model_effect(
    state: State,
    term: Model,
    prompt: str,
) -> None:
    if state.model_backend is None:
        raise EvaluationError(
            "This program performs a model effect, "
            "but no model backend was supplied.",
            term,
        )

    visible_stack = stack_string(state.data)
    if state.verbose:
        print("\n=== Model effect ===")
        print(f"Task: {prompt}")
        print(f"Stack (bottom to top): {visible_stack or '(empty)'}")

    turn = state.model_backend.generate(
        prompt=prompt,
        stack=visible_stack,
    )

    if state.verbose and turn.thinking and not turn.streamed:
        print("\n--- Model thinking ---")
        print(turn.thinking)

    if state.verbose:
        print("\n--- Model final answer ---")
        print(turn.answer)

    reply = parse_model_answer(turn.answer)

    if isinstance(reply, ModelError):
        raise ModelReportedError(prompt, reply.message)

    if state.verbose:
        print("\n--- Generated Copycat ---")
        print(reply.code or "(empty program)")

    try:
        generated = read(reply.code, source_name="<model output>")
    except CopycatError as exc:
        raise GeneratedCodeError(
            prompt,
            reply.code,
            exc,
        ) from exc

    if not state.allow_nested_model_calls and _contains_model(generated):
        raise GeneratedCodeError(
            prompt,
            reply.code,
            EvaluationError(
                "Generated code contains another model invocation, "
                "but nested model calls are disabled."
            ),
        )

    # `code` is a stack whose last element executes next. Appending the
    # generated program therefore replaces the model form in-place.
    state.code.append(generated)


In [ ]:
def evaluate(
    program: Object,
    dictionary: Optional[Mapping[str, Object]] = None,
    *,
    gas: int = 1_000_000,
    model_backend: Optional[ModelBackend] = None,
    strict: bool = False,
    allow_nested_model_calls: bool = False,
    verbose: bool = True,
) -> Object:
    primitives: dict[str, Primitive] = {
        "Copy": op_copy,
        "Drop": op_drop,
        "Swap": op_swap,
        "Abs": op_abs,
        "App": op_app,
        "Cat": op_cat,
        "Jump": op_jump,
        "Mark": op_mark,
    }

    state = State(
        code=[program],
        data=[],
        sink=[],
        gas=gas,
        dictionary=dictionary or {},
        primitives=primitives,
        model_backend=model_backend,
        strict=strict,
        allow_nested_model_calls=allow_nested_model_calls,
        verbose=verbose,
    )

    if verbose:
        print("=== Copycat evaluation ===")
        print(f"Program: {program or '(empty program)'}")
        print(f"Initial gas: {gas}")

    while state.is_active:
        term = state.tick()
        state.trace("before", term)

        match term:
            case Word(name):
                if binding := state.primitives.get(name):
                    binding(state)
                elif binding := state.dictionary.get(name):
                    state.code.append(binding)
                else:
                    state.fail_or_thunk(
                        f"Undefined word {name!r}.",
                        stop=True,
                    )

            case Abs(_) | Number(_) | String(_):
                state.data.append(term)

            case Cat(body):
                state.code.extend(reversed(body))

            case Model(prompt):
                _run_model_effect(state, term, prompt)

            case _:
                raise EvaluationError(
                    f"Unknown runtime object {term!r}.",
                    term,
                )

        state.trace("after", term)

    if state.gas <= 0 and state.code and state.strict:
        raise EvaluationError(
            "Evaluation ran out of gas before the program finished.",
            state.hand,
        )

    if verbose:
        print("\n=== Evaluation complete ===")
        print(f"Result: {state.value or '(empty)'}")

    return state.value


In [ ]:
def run(
    source: str,
    *,
    model_backend: Optional[ModelBackend] = None,
    strict: bool = False,
    dictionary: Optional[Mapping[str, Object]] = None,
    verbose: bool = True,
) -> str:
    return str(
        evaluate(
            read(source),
            dictionary=dictionary,
            model_backend=model_backend,
            strict=strict,
            verbose=verbose,
        )
    )


### Gemma 4 E2B backend


In [ ]:
COPYCAT_SYSTEM_PROMPT = r"""
You are the program-synthesis engine for Copycat, a tiny concatenative stack language.

You will receive:
1. the current Copycat data stack, written from bottom to top;
2. a natural-language task.

Your job is to synthesize the smallest Copycat program that performs the task
when executed immediately against that stack.

RETURN FORMAT
Your final response must contain exactly ONE protocol element:

<OK>Copycat code</OK>

or, only if you cannot produce a valid program:

<ERROR>a short, useful explanation</ERROR>

The protocol tag names are case-insensitive. Text outside that one element is
allowed, but do not emit a second OK or ERROR element. Do not put attributes on
the protocol element. If XML metacharacters occur inside the payload, escape
them correctly.

COPYCAT SYNTAX
Natural numbers:
  0
  1
  42

Strings:
  "hello"
  "two words"

Quotations:
  [Copy]
  [1 2 Swap]

Words are separated by whitespace.

PRIMITIVES
Copy
  Duplicate the top data-stack value.
  Example: 1 Copy  ==>  1 1

Drop
  Remove the top data-stack value.
  Example: 1 2 Drop  ==>  1

Swap
  Exchange the top two data-stack values.
  Example: 1 2 Swap  ==>  2 1

Abs
  Wrap the top value in a quotation.
  Example: 1 Abs  ==>  [1]

App
  Remove the top quotation and execute its contents.
  Example: 1 [Copy] App  ==>  1 1

Cat
  Concatenate the top two quotations.
  Example: [1] [2] Cat  ==>  [1 2]

Jump / Mark
  Delimited-control primitives. Do not use them unless the task actually
  requires continuation capture.

MODEL FORMS
Copycat also has {natural language} model forms, but DO NOT emit model forms
inside generated code in this version.

IMPORTANT
The generated code executes immediately with the current stack already present.
Do not reproduce existing stack values unless the task requires copying them.
Prefer the shortest valid program.

EXAMPLES

Current data stack:
1 2

Task:
swap the top two values

Final answer:
<OK>Swap</OK>


Current data stack:
"hello"

Task:
duplicate the top value

Final answer:
<OK>Copy</OK>


Current data stack:
(empty)

Task:
put the number 7 on the stack

Final answer:
<OK>7</OK>


Current data stack:
1

Task:
remove the value

Final answer:
<OK>Drop</OK>
""".strip()


In [ ]:
class Gemma4Backend:
    MODEL_ID = "google/gemma-4-E2B-it"

    def __init__(
        self,
        model,
        processor,
        *,
        system_prompt: str = COPYCAT_SYSTEM_PROMPT,
        max_new_tokens: int = 8_192,
        stream_output: bool = True,
    ):
        self.model = model
        self.processor = processor
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.stream_output = stream_output
        self.last_turn: Optional[ModelTurn] = None

    @classmethod
    def load(
        cls,
        model_id: str = MODEL_ID,
        *,
        system_prompt: str = COPYCAT_SYSTEM_PROMPT,
        max_new_tokens: int = 8_192,
        stream_output: bool = True,
    ) -> "Gemma4Backend":
        import torch
        from transformers import AutoModelForMultimodalLM, AutoProcessor

        processor = AutoProcessor.from_pretrained(model_id)

        model = AutoModelForMultimodalLM.from_pretrained(
            model_id,
            dtype=torch.float16,
            device_map="auto",
            attn_implementation="sdpa",
        )
        model.eval()

        return cls(
            model,
            processor,
            system_prompt=system_prompt,
            max_new_tokens=max_new_tokens,
            stream_output=stream_output,
        )

    def generate(self, *, prompt: str, stack: str) -> ModelTurn:
        import torch
        from transformers import TextStreamer

        visible_stack = stack if stack else "(empty)"

        messages = [
            {
                "role": "system",
                "content": self.system_prompt,
            },
            {
                "role": "user",
                "content": (
                    "Current data stack (bottom to top):\n"
                    f"{visible_stack}\n\n"
                    "Task:\n"
                    f"{prompt}"
                ),
            },
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            enable_thinking=True,
        ).to(self.model.device)

        input_len = inputs["input_ids"].shape[-1]
        streamer = None

        if self.stream_output:
            tokenizer = getattr(self.processor, "tokenizer", self.processor)
            streamer = TextStreamer(
                tokenizer,
                skip_prompt=True,
                skip_special_tokens=False,
            )
            print("\n--- Gemma live stream: thinking followed by final answer ---")

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=1.0,
                top_p=0.95,
                top_k=64,
                streamer=streamer,
            )

        if self.stream_output:
            print("--- End Gemma live stream ---")

        raw = self.processor.decode(
            outputs[0][input_len:],
            skip_special_tokens=False,
        )

        # Current Transformers versions accept the prompt tokens as a prefix
        # when parsing a response. The fallback keeps the notebook usable with
        # the earlier Gemma-4 model-card interface as well.
        try:
            parsed = self.processor.parse_response(
                raw,
                prefix=inputs["input_ids"],
            )
        except TypeError:
            parsed = self.processor.parse_response(raw)

        answer = parsed.get("content", "")
        thinking = parsed.get("thinking")

        if not isinstance(answer, str):
            answer = str(answer)

        if thinking is not None and not isinstance(thinking, str):
            thinking = str(thinking)

        turn = ModelTurn(
            answer=answer,
            thinking=thinking,
            raw=raw,
            streamed=self.stream_output,
        )
        self.last_turn = turn
        return turn


## Tests


In [ ]:
import ipytest
import pytest

ipytest.autoconfig()


In [ ]:
%%ipytest -q


@pytest.mark.parametrize(
    "source, expected",
    [
        ("[foo] Copy", "[foo] [foo]"),
        ("[foo] Drop", ""),
        ("[foo] [bar] Swap", "[bar] [foo]"),
        ("[foo] [bar] Cat", "[foo bar]"),
        ("[foo] Abs", "[[foo]]"),
        ("[foo] App", "foo"),
        ("[foo] Jump bar qux Mark baz", "[bar qux] foo baz"),
    ],
)
def test_original_examples(source, expected):
    assert run(source, verbose=False) == expected


def test_jump_finds_mark_when_mark_is_the_final_instruction():
    assert run("[App] Jump 1 Mark", verbose=False) == "1"


def test_model_form_is_opaque_to_copycat_syntax():
    program = read('{write [this] and "that"\non two lines}')
    (model,) = program.body
    assert isinstance(model, Model)
    assert model.prompt == 'write [this] and "that"\non two lines'


@pytest.mark.parametrize(
    "source, fragment",
    [
        ("[Copy", "Unclosed quotation"),
        ("Copy]", "no matching '['"),
        ("{do something", "Unclosed model invocation"),
        ('"unterminated', "unterminated string"),
        ("Copy @", "Unexpected character '@'"),
    ],
)
def test_parser_errors_are_explanatory(source, fragment):
    with pytest.raises(ParseError) as caught:
        read(source)
    assert fragment.lower() in str(caught.value).lower()


def test_strict_evaluation_reports_stack_underflow():
    with pytest.raises(EvaluationError) as caught:
        run("Copy", strict=True, verbose=False)

    message = str(caught.value)
    assert "Copy needs 1 value" in message
    assert "line 1, column 1" in message


def test_stub_model_ok_executes_generated_code_immediately():
    backend = StubModel("<OK>Swap</OK>")

    assert run(
        "1 2 {swap the top two values}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "2 1"

    assert backend.calls == [
        ("swap the top two values", "1 2")
    ]


def test_stub_model_error_becomes_structured_condition():
    backend = StubModel("<ERROR>I cannot do that safely.</ERROR>")

    with pytest.raises(ModelReportedError):
        run(
            "1 {do something impossible}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_protocol_allows_surrounding_text_and_lowercase_tags():
    backend = StubModel("I chose this.\n<ok>Copy</ok>\nDone.")

    assert run(
        "1 {duplicate the value}",
        model_backend=backend,
        strict=True,
        verbose=False,
    ) == "1 1"


def test_protocol_rejects_multiple_expected_elements():
    backend = StubModel("<OK>Copy</OK> or <ERROR>unsure</ERROR>")

    with pytest.raises(ModelProtocolError):
        run(
            "1 {duplicate the value}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


def test_bad_generated_syntax_is_attributed_to_model_output():
    backend = StubModel("<OK>[Copy</OK>")

    with pytest.raises(GeneratedCodeError) as caught:
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )

    assert "<model output>" in str(caught.value)


def test_nested_model_calls_are_disabled_by_default():
    backend = StubModel("<OK>{ask again}</OK>")

    with pytest.raises(GeneratedCodeError):
        run(
            "1 {do it}",
            model_backend=backend,
            strict=True,
            verbose=False,
        )


## Examples


Run this section in Colab to exercise the reader, deterministic model effects,
then load Gemma and run the live examples. Evaluator tracing is intentionally
enabled in the executable examples.


### Reader examples


In [ ]:
for source in [
    "1 2 Swap",
    "[foo] Jump bar qux Mark baz",
    '{write [this] and "that" on two lines}',
]:
    print(f"{source!r} -> {read(source)!r}")


### Deterministic model examples


In [ ]:
swap_stub = StubModel("Reasoning outside the tag is accepted.\n<ok>Swap</ok>")
copy_stub = StubModel("<OK>Copy</OK>")

print(
    run(
        "1 2 {put the top two values in the opposite order}",
        model_backend=swap_stub,
        strict=True,
    )
)

print(
    run(
        '"hello" {duplicate the top value}',
        model_backend=copy_stub,
        strict=True,
    )
)


### Live Gemma 4 synthesis


#### Load Gemma


In [ ]:
# Optional, only if your Hugging Face environment asks for authentication:
# from huggingface_hub import notebook_login
# notebook_login()

gemma = Gemma4Backend.load(
    max_new_tokens=8_192,
    stream_output=True,
)


#### Run live examples


In [ ]:
examples = [
    "1 2 {put the top two values in the opposite order}",
    '"hello" {duplicate the top value}',
    "{put the number 7 on the stack}",
]

for source in examples:
    print("\n" + "=" * 72)
    print("SOURCE:", source)
    try:
        print(
            "RESULT:",
            run(
                source,
                model_backend=gemma,
                strict=True,
            ),
        )
    except CopycatError as exc:
        print(exc)


## Future work

This version remains deliberately narrow: one model turn synthesizes a small
Copycat program, that program is parsed, and ordinary evaluation continues.
Repair loops, generic effect handlers, capabilities, external services,
simulation, persisted continuations, actors, and nested model effects remain
deferred until this one-shot path has been exercised with the live checkpoint.
